# ✈️ Airline Revenue Management Dashboard - Complete Walkthrough

This notebook provides a comprehensive guide to understanding and working with the **Qatar Airways RM Demo Dashboard**.

---

## 📋 Table of Contents
1. [Project Overview](#overview)
2. [Data Architecture](#data)
3. [ML Models & Algorithms](#models)
4. [Dashboard Visuals](#visuals)
5. [Business Problem & Solution](#business)
6. [How to Run & Work With the App](#howto)

---
<a id='overview'></a>
## 1. 🎯 Project Overview

### What is this Dashboard?
A **real-time decision support system** for airline revenue management that:
- Forecasts demand using S-curve booking pace models
- Optimizes pricing with elasticity analysis
- Predicts no-shows for overbooking decisions
- Provides AI-powered recommendations via RAG

### Tech Stack
| Layer | Technology |
|-------|------------|
| Frontend | React 19 + TypeScript |
| Build | Vite 6 |
| Styling | TailwindCSS |
| Charts | Recharts |
| State | TanStack React Query |
| AI | Google Gemini API |

---
<a id='data'></a>
## 2. 🗃️ Data Architecture

### Data Flow
```
mockData.ts → API Layer → React Query → Dashboard Components
```

### Core Data Types (from `types.ts`)

In [ ]:
# Key Data Structures

# 1. Flight Data
flight = {
    "id": "f1",
    "flightNumber": "QR701",
    "origin": "DOH",
    "destination": "JFK",
    "capacity": 354,
    "booked": 280,
    "currentLoadFactor": 79.1,
    "rask": 10.8,  # Revenue per Available Seat Km
    "yield": 13.5,  # Revenue per Passenger Km
    "status": "Opportunity"  # On Track | Critical | Overbooked | Opportunity
}

# 2. Route KPIs
route_kpi = {
    "route": "DOH-JFK",
    "loadFactor": 88,
    "targetLoadFactor": 92,
    "rask": 12.4,
    "raskTrend": 1.5,  # % change
}

# 3. Booking Curve Point (S-Curve)
booking_point = {
    "daysOut": 60,
    "actual": 180,
    "forecast": 200,
    "ly": 165  # Last Year comparison
}

### Routes Covered
The dashboard focuses on 5 high-potential routes:

| Route | Characteristics |
|-------|----------------|
| DOH-SFO | High growth (Tech corridor) |
| DOH-JFK | Premium leisure/business mix |
| DOH-LOS | High yield, volatile load |
| DOH-PVG | Emerging market, high growth |
| DOH-ZAG | Event-driven demand |

In [ ]:
# Sample Route KPI Data (from mockData.ts)

MOCK_ROUTE_KPIS = {
    'DOH-SFO': {'loadFactor': 82, 'targetLoadFactor': 90, 'rask': 9.8, 'raskTrend': 4.2},
    'DOH-JFK': {'loadFactor': 88, 'targetLoadFactor': 92, 'rask': 12.4, 'raskTrend': 1.5},
    'DOH-LOS': {'loadFactor': 74, 'targetLoadFactor': 85, 'rask': 14.5, 'raskTrend': -2.1},
    'DOH-PVG': {'loadFactor': 65, 'targetLoadFactor': 80, 'rask': 7.2, 'raskTrend': 8.5},
    'DOH-ZAG': {'loadFactor': 79, 'targetLoadFactor': 85, 'rask': 8.9, 'raskTrend': 0.5},
}

print("Route Performance Summary:")
for route, kpis in MOCK_ROUTE_KPIS.items():
    gap = kpis['targetLoadFactor'] - kpis['loadFactor']
    print(f"{route}: Load {kpis['loadFactor']}% (Gap: {gap}%), RASK Trend: {kpis['raskTrend']:+.1f}%")

---
<a id='models'></a>
## 3. 🤖 ML Models & Algorithms

### Model 1: S-Curve Demand Forecasting

In [ ]:
import numpy as np

def booking_forecast(t, L=350, k=10, t0=0.5):
    """
    S-Curve Logistic Function for booking pace prediction.
    
    P(t) = L / (1 + e^(-k(t-t0)))
    
    Args:
        t: Progress (0 to 1, where 0=90 days out, 1=departure)
        L: Maximum capacity (asymptote)
        k: Steepness of curve
        t0: Midpoint of curve
    """
    return L / (1 + np.exp(-k * (t - t0)))

# Generate sample booking curve
days = np.linspace(0, 1, 15)
bookings = [booking_forecast(t) for t in days]
print("S-Curve Booking Forecast (Days 90→0):")
print([f"{int(b)}" for b in bookings])

### Model 2: Price Elasticity Estimation

In [ ]:
def calculate_revenue_impact(price_change_pct, elasticity=-1.5):
    """
    Price Elasticity of Demand (PED) model.
    
    ε = (% Change in Quantity) / (% Change in Price)
    
    If |ε| > 1: Elastic demand (price cuts increase revenue)
    If |ε| < 1: Inelastic demand (price increases boost revenue)
    """
    volume_change = 1 + (elasticity * price_change_pct / 100)
    price_factor = 1 + (price_change_pct / 100)
    revenue_change = (price_factor * volume_change - 1) * 100
    return round(revenue_change, 2)

# Elasticity scenarios (from mockData.ts)
scenarios = [-10, -5, 0, 5, 10]
print("Price Change → Revenue Impact:")
for pct in scenarios:
    impact = calculate_revenue_impact(pct)
    print(f"  {pct:+d}% price → {impact:+.1f}% revenue")

### Model 3: Overbooking Optimization (Newsvendor)

In [ ]:
def simulate_overbooking(capacity, overbook, no_show_rate=0.05, avg_fare=800, dbc_cost=400):
    """
    Monte Carlo simulation for overbooking decisions.
    
    Newsvendor Model balances:
    - Cu (Underage): Cost of empty seat (spoilage)
    - Co (Overage): Cost of denied boarding compensation
    """
    np.random.seed(42)
    n_sims = 1000
    results = []
    
    for _ in range(n_sims):
        show_ups = np.random.binomial(capacity + overbook, 1 - no_show_rate)
        denied = max(0, show_ups - capacity)
        revenue = min(show_ups, capacity) * avg_fare - denied * dbc_cost
        results.append(revenue)
    
    return {'mean': np.mean(results), 'std': np.std(results)}

# Overbooking scenarios
print("Overbooking Analysis (300 capacity, 5% no-show):")
for ob in [2, 4, 6, 8, 10]:
    result = simulate_overbooking(300, ob)
    print(f"  +{ob} seats: ${result['mean']:,.0f} avg revenue")

### Model 4: RAG-based AI Assistant

The RM Assistant uses **Retrieval-Augmented Generation**:

```python
# RAG Pipeline
1. User Query → Embedding (Sentence Transformer)
2. Cosine Similarity → Retrieve Top-K Documents
3. Context + Query → Gemini API → Response
4. Faithfulness Scoring → Citation Verification
```

**Faithfulness Score**: Measures how well the AI response is grounded in source documents (target: >90%)

---
<a id='visuals'></a>
## 4. 📊 Dashboard Visuals

### Visual Components Overview

| Visual | Component | Business Question |
|--------|-----------|------------------|
| Load Factor Gauge | `StrategicDashboard.tsx` | Are we filling seats at target revenue? |
| Booking S-Curve | `DynamicPricingPanel.tsx` | Are bookings ahead/behind forecast? |
| Competitor Tracker | `StrategicDashboard.tsx` | Are we priced competitively? |
| Waterfall Chart | `StrategicDashboard.tsx` | What's driving profit/loss? |
| Elasticity Scatter | `UnconstrainingPanel.tsx` | How sensitive is demand to price? |
| Overbooking Bars | `NoShowPanel.tsx` | How aggressive can we overbook? |
| RAG Faithfulness | `Assistant.tsx` | Can we trust AI recommendations? |

In [ ]:
# Sample Waterfall Data (Route Profitability)

waterfall_doh_sfo = [
    {'name': 'Base Fare', 'value': 45000, 'type': 'total'},
    {'name': 'Fuel Surcharge', 'value': 12000, 'type': 'increase'},
    {'name': 'Ancillaries', 'value': 5000, 'type': 'increase'},
    {'name': 'Spoilage', 'value': -4000, 'type': 'decrease'},
    {'name': 'Spill Cost', 'value': -1500, 'type': 'decrease'},
    {'name': 'Ops Cost', 'value': -25000, 'type': 'decrease'},
    {'name': 'Net Profit', 'value': 31500, 'type': 'total'}
]

print("DOH-SFO Profit Decomposition:")
for item in waterfall_doh_sfo:
    sign = '+' if item['value'] > 0 and item['type'] != 'total' else ''
    print(f"  {item['name']}: {sign}${item['value']:,}")

---
<a id='business'></a>
## 5. 💼 Business Problem & Solution

### The Core Challenge

> **"Anticipating demand spikes from events (World Cup, Expo) or disruptions (fuel shocks) to prevent over/under-pricing."**

### How Each Visual Solves a Problem

| Problem | Visual | Action | Revenue Impact |
|---------|--------|--------|---------------|
| High demand but low yield | Load Factor Gauge | Close discount buckets | +$15k/flight |
| Premature sell-out risk | Booking S-Curve | Restrict discount classes | +$25k saved |
| Competitor price drop | Price Tracker | Hold price if demand strong | +$10k saved |
| Unsold premium seats | Waterfall | Trigger T-24h upgrades | +$4k profit |
| Weak load factor | Elasticity Scatter | Strategic discount | +6.2% revenue |
| Full flight + no-shows | Overbooking Model | Authorize +5 seats | +$3.8k net |

---
<a id='howto'></a>
## 6. 🛠️ How to Run & Work With the App

### Quick Start

```bash
# 1. Clone repository
git clone https://github.com/your-username/AirlineDashboard.git
cd AirlineDashboard

# 2. Install dependencies
npm install --legacy-peer-deps

# 3. Configure API key
# Create .env.local with:
GEMINI_API_KEY=your_key_here

# 4. Start dev server
npm run dev

# 5. Open http://localhost:3000
```

### Project Structure

```
AirlineDashboard/
├── App.tsx                 # Main app with routing
├── types.ts                # TypeScript interfaces
├── components/
│   ├── dashboard/
│   │   ├── StrategicDashboard.tsx  # Main dashboard
│   │   ├── DynamicPricingPanel.tsx # Demand forecasting
│   │   ├── NoShowPanel.tsx         # Overbooking optimizer
│   │   ├── UnconstrainingPanel.tsx # Pricing optimizer
│   │   └── Assistant.tsx           # AI chat interface
│   ├── layout/
│   │   ├── Sidebar.tsx
│   │   └── Header.tsx
│   └── ui/                 # Reusable components
├── services/
│   ├── mockData.ts         # Mock data & API
│   └── api/
│       ├── config.ts       # API configuration
│       └── cache.ts        # Caching layer
└── lib/
    └── utils.ts            # Utility functions
```

### Dashboard Views

Navigate via sidebar:

1. **Dashboard** - Strategic KPIs (Load Factor, RASK, Yield gauges)
2. **Demand Forecasting** - Interactive S-curve analysis
3. **No-Show Predictor** - Overbooking risk optimization
4. **Pricing Optimizer** - Elasticity analysis & unconstraining
5. **RM Assistant** - AI-powered Q&A with RAG

### Working with the Code

#### Adding a New Route
```typescript
// In services/mockData.ts
MOCK_ROUTE_KPIS['DOH-NEW'] = {
    id: '6', route: 'DOH-NEW',
    loadFactor: 75, targetLoadFactor: 85,
    rask: 10.0, raskTrend: 2.0,
    yield: 12.0, yieldTrend: 1.0
};
```

#### Modifying the AI Assistant
```typescript
// In components/dashboard/Assistant.tsx
// Update the system prompt or add new capabilities
```

#### Customizing Visualizations
```typescript
// All charts use Recharts library
// See StrategicDashboard.tsx for examples
```

---

## 📚 Summary

This dashboard demonstrates:

✅ **Predictive Modeling** - S-curve forecasting, demand prediction  
✅ **Dynamic Pricing** - Elasticity analysis, optimization  
✅ **AI/ML Implementation** - RAG, LLM integration  
✅ **Data Engineering** - Schema design, API development  
✅ **Production Code** - TypeScript, React, modern tooling

---

*Dashboard live at: http://localhost:3000*